# YapIndex RAG Demo

This notebook runs the complete pipeline:

`question -> FAISS GPU retrieval -> BGE reranking -> Qwen answer -> sources`

Select the **YapIndex WSL GPU** kernel before running the cells.

## Step 1: Check the GPU environment

In [ ]:
import sys
from pathlib import Path

import faiss
import torch

PROJECT_ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "utils").is_dir()
)
sys.path.insert(0, str(PROJECT_ROOT))

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select YapIndex WSL GPU.")

print(f"Python: {sys.executable}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"FAISS GPUs: {faiss.get_num_gpus()}")

## Step 2: Load FAISS, reranker, and generator

In [ ]:
from utils.faiss_store import load_vectorstore
from utils.generator import load_generator
from utils.reranker import RERANKER_MODEL, load_reranker

print("Loading FAISS GPU index...")
vectorstore = load_vectorstore(
    index_path=str(PROJECT_ROOT / "faiss_index"),
    use_gpu=True,
)

print(f"Loading reranker: {RERANKER_MODEL}")
reranker = load_reranker(device="cuda")
print(f"Reranker device: {reranker.device}")

print("Loading answer generator...")
generator = load_generator()
print("Models loaded.")

## Step 3: Evaluate the complete set

This cell runs all questions in `evals/meridian_rag_eval.jsonl`. It may take a while because the local LLM generates one answer per question.

In [4]:
import json

from utils.rag import answer_question

questions_path = PROJECT_ROOT / "evals" / "meridian_rag_eval.jsonl"
with questions_path.open(encoding="utf-8") as file:
    questions = [json.loads(line) for line in file if line.strip()]

print(f"Evaluating {len(questions)} questions.")

results = []
for index, question in enumerate(questions, start=1):
    print(
        f"Evaluating {index}/{len(questions)}: {question['id']}",
        end="\r",
        flush=True,
    )

    answer, documents = answer_question(
        vectorstore,
        generator,
        question["question"],
        retrieval_k=10,
        final_k=5,
        reranker=reranker,
    )

    sources = [
        document.metadata.get("source", "Unknown").replace("\\", "/")
        for document in documents
    ]
    answer_lower = answer.lower()
    facts = question.get("answer_facts", [])
    fact_coverage = (
        sum(fact.lower() in answer_lower for fact in facts) / len(facts)
        if facts
        else 0.0
    )
    gold_sources = set(question.get("gold_sources", []))
    source_hit = bool(gold_sources.intersection(sources)) if gold_sources else True

    refusal_markers = (
        "does not contain enough information",
        "do not know",
        "not enough information",
        "cannot answer",
    )
    refused = any(marker in answer_lower for marker in refusal_markers)

    results.append(
        {
            "id": question["id"],
            "category": question["category"],
            "question": question["question"],
            "answer": answer,
            "sources": sources,
            "fact_coverage": fact_coverage,
            "source_hit": source_hit,
            "refused": refused,
        }
    )

print("\nEvaluation complete.")

Evaluating 70 questions.
Reranking 10 documents...
Reranking 10 documents...
Reranking 10 documents...
Reranking 10 documents...
Reranking 10 documents...
Reranking 10 documents...
Reranking 10 documents...
Reranking 10 documents...
Reranking 10 documents...
Reranking 10 documents...
Reranking 10 documents...
Reranking 10 documents...
Reranking 10 documents...
Reranking 10 documents...
Reranking 10 documents...
Reranking 10 documents...
Reranking 10 documents...
Reranking 10 documents...
Reranking 10 documents...
Reranking 10 documents...
Reranking 10 documents...
Reranking 10 documents...
Reranking 10 documents...
Reranking 10 documents...
Reranking 10 documents...
Reranking 10 documents...
Reranking 10 documents...
Reranking 10 documents...
Reranking 10 documents...
Reranking 10 documents...
Reranking 10 documents...
Reranking 10 documents...
Reranking 10 documents...
Reranking 10 documents...
Reranking 10 documents...
Reranking 10 documents...
Reranking 10 documents...
Reranking 10 

## Step 4: Review results

The summary shows answer fact coverage, gold-source retrieval, refusal behavior, and the weakest answers.

In [5]:
count = len(results)
mean_fact_coverage = sum(item["fact_coverage"] for item in results) / count
source_hit_rate = sum(item["source_hit"] for item in results) / count

unanswerable = [item for item in results if item["category"] == "unanswerable"]
refusal_rate = (
    sum(item["refused"] for item in unanswerable) / len(unanswerable)
    if unanswerable
    else 0.0
)

print(f"Questions evaluated: {count}")
print(f"Mean answer fact coverage: {mean_fact_coverage:.2%}")
print(f"Gold-source hit rate: {source_hit_rate:.2%}")
print(f"Unanswerable refusal rate: {refusal_rate:.2%}")

print("\nWeakest answers:")
for item in sorted(results, key=lambda result: result["fact_coverage"])[:5]:
    print(f"\n{item['id']} ({item['fact_coverage']:.0%} facts): {item['question']}")
    print(f"Answer: {item['answer']}")
    print(f"Sources: {item['sources']}")

Questions evaluated: 70
Mean answer fact coverage: 54.98%
Gold-source hit rate: 78.57%
Unanswerable refusal rate: 100.00%

Weakest answers:

Q-005 (0% facts): What container orchestrator runs Meridian's production services?
Answer: The knowledge base does not contain enough information.

The provided context does not mention which container orchestrator runs Meridian's production services. It only describes the deployment process using ArgoCD for the current flow and mentions that previously Jenkins was used, but it does not specify the orchestrator for the production services.
Sources: ['company/runbooks/deploy-legacy-jenkins.md', 'company/runbooks/deploy-current.md', 'company/cicd/release-process.md', 'company/cicd/argocd-setup.md', 'company/infrastructure/eks-cluster.md']

Q-015 (0% facts): How does a new engineer join the on-call rotation at Meridian?
Answer: The knowledge base does not contain enough information.

The provided context does not specify the exact process for a new e